## 实验3：常量成员函数

本实验区分返回类型中的 `const` 与成员函数末尾的 `const`，并观察只读引用为什么只能调用常量成员函数。完成后应能够为 getter 选择值返回或引用返回，并说明相应的生命周期约束。

### 实验代码

In [1]:
#include <iostream>
#include <string>

In [2]:
class User {
public:
    User(
        std::string name,
        int age
    )
        : name_(name),
          age_(age) {
    }

    const std::string& name() const {
        return name_;
    }

    int age() const {
        return age_;
    }

    void set_age(int age) {
        age_ = age;
    }

private:
    std::string name_;
    int age_;
};

void print_user(
    const User& user
) {
    std::cout
        << user.name()
        << ", "
        << user.age()
        << '\n';
}


{
    User user(
        "Bob",
        20
    );

    print_user(user);

    user.set_age(21);

    print_user(user);
}


Bob, 20
Bob, 21


这里第一次出现常量成员函数：
```C++
int age() const
```
末尾的 `const` 是函数类型的一部分，表示该函数不会通过 `this` 修改对象的可观察状态。对调用者而言，这是一项可以由编译器检查的 API 承诺。

---


### 两个`const`不是一回事
例如：
```C++
const std::string& name() const;
```
1. 第一个 `const` 修饰 `std::string`，表示通过返回的引用不能修改字符串。
2. `&` 表示返回引用而不是创建新的字符串对象。
3. 函数末尾的 `const` 修饰隐式的 `this` 所指对象，表示该成员函数不会修改对象的可观察状态。

因此，这两个 `const` 分别约束“调用者能否通过返回值修改成员”和“成员函数能否修改当前对象”，不能互相替代。



---


### 为什么必须写`const`成员函数
这里：
```C++
void print_user(
    const User& user
);
}
```
`user` 是一个 `User` 的只读引用，所以不能调用普通的 `int age()`：没有末尾 `const` 的成员函数允许修改对象，编译器不能假设它是只读操作。

而
```C++
int age() const;
```
是在 API 规范中承诺：
```
调用 age()
但是不会修改 User
```
于是 `const User&` 才能安全调用。相同的常量成员函数也可以由非常量对象调用，因此只读查询通常都应声明为 `const`。

---


### getter 不一定要返回对象拷贝
观察：
```C++
const std::string& name() const{
    return name_;
}
```

为什么不是：
```C++
std::string name() const{
    return name_;
}
```

- 前者：
```
User
 └── name_

      ▲
      │
const reference
```
没有字符串拷贝，同时调用者不能通过该引用修改私有字段的内容。
所以：
```C++
const std::string&
```
表示：借给你看，但是不允许你修改

不过，不是所有 getter 都应返回引用，必须结合对象生命周期判断：

- 返回引用的有效期不会超过所属 `User` 对象及其 `name_` 成员。
- 如果 `User` 已销毁，先前取得的引用会成为悬空引用。
- 对 `int` 这样的小型标量，按值返回更简单且成本很低，所以 `age()` 返回 `int`。
- 当调用者需要独立拥有结果，或返回值生命周期难以保证时，应优先按值返回。

### 预期输出

```text
Bob, 20
Bob, 21
```

第一次打印读取初始状态，`set_age(21)` 修改对象后第二次打印读取新状态。`print_user` 只能观察对象，不能通过其 `const User&` 参数修改对象。

### 实验结论

常量成员函数提供可检查的只读接口；返回 `const T&` 可以避免拷贝，但会把返回值的有效期绑定到原对象。设计 Native SDK 或 C++ API 时，性能收益必须与 ownership 和 lifetime 的清晰程度一起权衡。